# CRTB Simulation Study

Evaluate **crtb** (Cellwise Robust Twoblock) against **twoblock** and **rtb** under
cellwise contamination in both the X and Y blocks.

## Contamination design
Inherited from the CRM simulation:
- A fixed fraction of rows (70%) contains at least one outlying cell.
- Within each affected row, cells are contaminated independently at a rate that
  achieves the target cell-contamination percentage over the whole block.
- Outlier magnitude: uniform ±10 shift on the raw data.
- Cell contamination levels tested: **10 %, 20 %, 30 %** (applied to both X and Y).
- Because 70 % of rows are contaminated, **more than half the rows are affected**
  at every contamination level — the scenario where casewise-robust methods break down.

## Dimensionality scenarios
Six (p_signal, p_noise) configurations as in `simulation_rtb.ipynb`, covering
p < n (p_signal = 20) and p > n (p_signal = 150) with varying proportions of
uninformative variables.

## Methods compared
| Label | Class | sparse |
|---|---|---|
| TB dense | `twoblock` | False |
| RTB dense | `rtb` | False |
| CRTB dense | `crtb` | False |
| TB sparse | `twoblock` | True |
| RTB sparse | `rtb` | True |
| CRTB sparse | `crtb` | True |

## Metrics
- $\text{MSE}(B) = \frac{1}{pq}\|\hat{B} - B_{\text{true}}\|_F^2$ — coefficient accuracy
- **F1 score** — variable selection accuracy (sparse methods, configs with noise variables)

Averaged over 200 repeats.

In [ ]:
import numpy as np
import pandas as ps
import warnings
warnings.filterwarnings('ignore')

from twoblock import twoblock, rtb, crtb

## Simulation parameters

In [ ]:
# Fixed parameters
n          = 100    # observations
k          = 3      # true latent dimension (= optimal n_components_x)
q          = 4      # Y variables
sigma_e    = 0.5    # X noise std
sigma_f    = 0.5    # Y noise std
outlier_shift = 10  # contamination magnitude
n_repeats  = 200
eta_x_sim  = 0.5    # sparsity parameter for sparse methods
eta_y_sim  = 0.5

# Row contamination rate: fixed at 70 % so that >50 % of rows are always affected
row_rate = 0.70

# Cell contamination levels to test (applied to both X and Y)
cell_cont_pcts = [0.0, 0.10, 0.20, 0.30]

# (p_signal, p_noise) dimensionality configurations
dim_configs = {
    'p20_noise0':    (20,  0),
    'p20_noise10':   (20,  10),
    'p20_noise80':   (20,  80),
    'p150_noise0':   (150, 0),
    'p150_noise50':  (150, 50),
    'p150_noise250': (150, 250),
}

print(f"n={n}, k={k}, q={q}, repeats={n_repeats}")
print(f"Row contamination rate: {row_rate*100:.0f}% (fixed)")
print(f"Cell contamination levels: {[f'{c*100:.0f}%' for c in cell_cont_pcts]}")
print(f"Dimensionality configs:")
for label, (ps_, pn) in dim_configs.items():
    print(f"  {label}: p_signal={ps_}, p_noise={pn}, p_total={ps_+pn}")

## Data generation and contamination functions

In [ ]:
def generate_clean_data(rng, n, p_signal, p_noise, k, q, sigma_e, sigma_f):
    """
    Generate clean two-block data from a latent variable model.

    X = T @ P_signal.T + E   (n x p, p = p_signal + p_noise)
    Y = T @ C + F            (n x q)
    B_true = [P_signal @ C; 0]  (p x q)

    P_signal is orthonormal (from QR), T ~ N(0,I), E ~ N(0, sigma_e^2 I),
    F ~ N(0, sigma_f^2 I), C ~ N(0,I).
    """
    p = p_signal + p_noise

    T = rng.standard_normal((n, k))

    P_raw = rng.standard_normal((p_signal, k))
    P_signal, _ = np.linalg.qr(P_raw)
    P_signal = P_signal[:, :k]

    C = rng.standard_normal((k, q))

    E = sigma_e * rng.standard_normal((n, p_signal))
    X_signal = T @ P_signal.T + E

    if p_noise > 0:
        X_noise = sigma_e * rng.standard_normal((n, p_noise))
        X = np.hstack([X_signal, X_noise])
    else:
        X = X_signal

    F = sigma_f * rng.standard_normal((n, q))
    Y = T @ C + F

    B_true = np.zeros((p, q))
    B_true[:p_signal, :] = P_signal @ C

    return X, Y, B_true


def add_cellwise_outliers_block(Z, cell_cont_pct, outlier_shift, rng, row_rate=0.70):
    """
    Add cellwise outliers to a data block Z (n x d).

    Strategy (from CRM simulation):
      - `row_rate` fraction of rows are selected for contamination.
      - Within each selected row, Poisson(cells_per_row) cells are contaminated,
        where cells_per_row = cell_cont_pct * d / row_rate targets the desired
        overall cell contamination percentage.
      - At least 1, at most d cells are contaminated per affected row.
      - Each outlying cell is shifted by ±outlier_shift (sign chosen at random).

    Returns Z_cont (contaminated copy) and outlier_mask (bool, n x d).
    """
    if cell_cont_pct == 0.0:
        return Z.copy(), np.zeros(Z.shape, dtype=bool)

    n, d = Z.shape
    cells_per_row = cell_cont_pct * d / row_rate

    Z_cont = Z.copy()
    outlier_mask = np.zeros((n, d), dtype=bool)

    n_cont_rows = int(np.ceil(n * row_rate))
    cont_rows = rng.choice(n, size=n_cont_rows, replace=False)

    for row in cont_rows:
        n_cells = max(1, min(d, rng.poisson(cells_per_row)))
        cont_cols = rng.choice(d, size=n_cells, replace=False)
        signs = rng.choice([-1, 1], size=n_cells)
        Z_cont[row, cont_cols] += signs * outlier_shift
        outlier_mask[row, cont_cols] = True

    return Z_cont, outlier_mask


def mse_B(B_est, B_true):
    """Mean squared error per element of the coefficient matrix."""
    return np.mean((B_est - B_true) ** 2)


def f1_score(indret, p_signal, p):
    """
    F1 score for X variable selection.

    Signal variables are indices 0 .. p_signal-1;
    noise variables are p_signal .. p-1.
    `indret` is the array of selected column indices (from x_indret_).
    """
    selected = set(indret.tolist())
    signal   = set(range(p_signal))
    tp = len(selected & signal)
    fp = len(selected - signal)
    fn = len(signal - selected)
    denom = 2 * tp + fp + fn
    return (2 * tp / denom) if denom > 0 else 0.0

## Verify contamination design

In [ ]:
rng_check = np.random.default_rng(42)
X_chk, Y_chk, _ = generate_clean_data(rng_check, n=100, p_signal=20, p_noise=10,
                                        k=3, q=4, sigma_e=0.5, sigma_f=0.5)

print(f"{'Cell cont. target':>20} {'Actual X cells':>16} {'Actual X rows':>14} {'Actual Y cells':>16} {'Actual Y rows':>14}")
print("-" * 85)
for ccp in [0.10, 0.20, 0.30]:
    rng_v = np.random.default_rng(0)
    Xc, mx = add_cellwise_outliers_block(X_chk, ccp, 10.0, rng_v, row_rate=0.70)
    Yc, my = add_cellwise_outliers_block(Y_chk, ccp, 10.0, rng_v, row_rate=0.70)
    xc_pct = 100 * mx.sum() / mx.size
    xr_pct = 100 * mx.any(axis=1).sum() / mx.shape[0]
    yc_pct = 100 * my.sum() / my.size
    yr_pct = 100 * my.any(axis=1).sum() / my.shape[0]
    print(f"{ccp*100:>19.0f}%  {xc_pct:>14.1f}%  {xr_pct:>12.1f}%  {yc_pct:>14.1f}%  {yr_pct:>12.1f}%")

## Single-repeat runner

In [ ]:
# Shared Hampel cutoffs — relaxed to handle heavy contamination
_probp1, _probp2, _probp3 = 0.75, 0.90, 0.95

def run_single(rng, n, p_signal, p_noise, k, q, sigma_e, sigma_f,
               cell_cont_pct, outlier_shift, row_rate=0.70):
    """
    Run one simulation repeat.

    Generates clean data, adds independent cellwise contamination to X and Y,
    fits six methods (dense and sparse variants of twoblock, rtb, crtb),
    and returns a dict of MSE(B) and F1 values.
    """
    X, Y, B_true = generate_clean_data(rng, n, p_signal, p_noise, k, q, sigma_e, sigma_f)
    Xc, _ = add_cellwise_outliers_block(X, cell_cont_pct, outlier_shift, rng, row_rate)
    Yc, _ = add_cellwise_outliers_block(Y, cell_cont_pct, outlier_shift, rng, row_rate)

    p        = p_signal + p_noise
    has_noise = p_noise > 0
    nc_y     = min(k, q)
    res      = {}

    # ---- twoblock dense ----
    try:
        tb = twoblock(n_components_x=k, n_components_y=nc_y,
                      scale='std', verbose=False, copy=False)
        tb.fit(Xc, Yc)
        res['tb_mse']  = mse_B(tb.coef_, B_true)
    except Exception:
        res['tb_mse']  = np.nan

    # ---- twoblock sparse ----
    try:
        tbs = twoblock(n_components_x=k, n_components_y=nc_y,
                       sparse=True, eta_x=eta_x_sim, eta_y=eta_y_sim,
                       scale='std', verbose=False, copy=False)
        tbs.fit(Xc, Yc)
        res['tbs_mse'] = mse_B(tbs.coef_, B_true)
        res['tbs_f1']  = f1_score(tbs.x_indret_, p_signal, p) if has_noise else np.nan
    except Exception:
        res['tbs_mse'] = np.nan
        res['tbs_f1']  = np.nan

    # ---- rtb dense ----
    try:
        r = rtb(n_components_x=k, n_components_y=nc_y,
                centre='median', scale='mad', verbose=False, copy=False,
                probp1=_probp1, probp2=_probp2, probp3=_probp3)
        r.fit(Xc, Yc)
        res['rtb_mse'] = mse_B(r.coef_, B_true)
    except Exception:
        res['rtb_mse'] = np.nan

    # ---- rtb sparse ----
    try:
        rs = rtb(n_components_x=k, n_components_y=nc_y,
                 sparse=True, eta_x=eta_x_sim, eta_y=eta_y_sim,
                 centre='median', scale='mad', verbose=False, copy=False,
                 probp1=_probp1, probp2=_probp2, probp3=_probp3)
        rs.fit(Xc, Yc)
        res['rtbs_mse'] = mse_B(rs.coef_, B_true)
        res['rtbs_f1']  = f1_score(rs.x_indret_, p_signal, p) if has_noise else np.nan
    except Exception:
        res['rtbs_mse'] = np.nan
        res['rtbs_f1']  = np.nan

    # ---- crtb dense ----
    try:
        cr = crtb(n_components_x=k, n_components_y=nc_y,
                  centre='median', scale='mad', verbose=False, copy=False,
                  probp1=_probp1, probp2=_probp2, probp3=_probp3)
        cr.fit(Xc, Yc)
        res['crtb_mse'] = mse_B(cr.coef_, B_true)
    except Exception:
        res['crtb_mse'] = np.nan

    # ---- crtb sparse ----
    try:
        crs = crtb(n_components_x=k, n_components_y=nc_y,
                   sparse=True, eta_x=eta_x_sim, eta_y=eta_y_sim,
                   centre='median', scale='mad', verbose=False, copy=False,
                   probp1=_probp1, probp2=_probp2, probp3=_probp3)
        crs.fit(Xc, Yc)
        res['crtbs_mse'] = mse_B(crs.coef_, B_true)
        res['crtbs_f1']  = f1_score(crs.x_indret_, p_signal, p) if has_noise else np.nan
    except Exception:
        res['crtbs_mse'] = np.nan
        res['crtbs_f1']  = np.nan

    return res

## Main simulation loop

Iterates over all (dim_config × cell_cont_pct) scenarios, running `n_repeats` repeats each.

In [ ]:
import time

rng = np.random.default_rng(42)

metrics = ['tb_mse', 'rtb_mse', 'crtb_mse',
           'tbs_mse', 'rtbs_mse', 'crtbs_mse',
           'tbs_f1', 'rtbs_f1', 'crtbs_f1']

scenarios = [
    (dim_label, p_signal, p_noise, ccp)
    for dim_label, (p_signal, p_noise) in dim_configs.items()
    for ccp in cell_cont_pcts
]

print(f"Total scenarios : {len(scenarios)}")
print(f"Fits per scenario: {n_repeats} repeats × 6 methods = {n_repeats * 6}")
print(f"Total model fits: {len(scenarios) * n_repeats * 6}")

all_results = []
t0 = time.time()

for si, (dim_label, p_signal, p_noise, ccp) in enumerate(scenarios):
    collected = {m: [] for m in metrics}

    for rep in range(n_repeats):
        res = run_single(rng, n, p_signal, p_noise, k, q, sigma_e, sigma_f,
                         ccp, outlier_shift, row_rate)
        for m in metrics:
            collected[m].append(res.get(m, np.nan))

    row = {
        'dim_config':    dim_label,
        'p_signal':      p_signal,
        'p_noise':       p_noise,
        'p_total':       p_signal + p_noise,
        'cell_cont_pct': ccp,
    }
    for m in metrics:
        arr = np.array(collected[m])
        row[f'{m}_mean'] = np.nanmean(arr)
        row[f'{m}_sd']   = np.nanstd(arr)
    all_results.append(row)

    elapsed = time.time() - t0
    print(f"  [{si+1:2d}/{len(scenarios)}] {dim_label}, cont={ccp*100:.0f}%  "
          f"TB={row['tb_mse_mean']:.4f}  RTB={row['rtb_mse_mean']:.4f}  "
          f"CRTB={row['crtb_mse_mean']:.4f}  ({elapsed:.0f}s)",
          flush=True)

print(f"\nDone in {time.time()-t0:.0f}s")

## Results table

In [ ]:
df = ps.DataFrame(all_results)

# Format helper: mean (sd)
def fmt(df, col):
    return df.apply(lambda r: f"{r[col+'_mean']:.4f} ({r[col+'_sd']:.4f})", axis=1)

# MSE table
print("=== MSE(B): mean (sd) over", n_repeats, "repeats ===")
mse_table = df[['dim_config', 'p_total', 'cell_cont_pct']].copy()
mse_table['cell_cont_pct'] = (mse_table['cell_cont_pct'] * 100).astype(int).astype(str) + '%'
for label, col in [('TB',   'tb_mse'),  ('RTB',  'rtb_mse'),  ('CRTB', 'crtb_mse'),
                    ('TBs',  'tbs_mse'), ('RTBs', 'rtbs_mse'), ('CRTBs','crtbs_mse')]:
    mse_table[label] = fmt(df, col)
display(mse_table)

# F1 table — sparse methods, noise configs only
noise_mask = df['p_noise'] > 0
print("\n=== Variable selection F1: sparse methods, configs with noise variables ===")
f1_table = df.loc[noise_mask, ['dim_config', 'p_signal', 'p_noise', 'cell_cont_pct']].copy()
f1_table['cell_cont_pct'] = (f1_table['cell_cont_pct'] * 100).astype(int).astype(str) + '%'
for label, col in [('TBs F1', 'tbs_f1'), ('RTBs F1', 'rtbs_f1'), ('CRTBs F1', 'crtbs_f1')]:
    f1_table[label] = fmt(df.loc[noise_mask], col)
display(f1_table)

## Visualisation — MSE bar charts

One panel per dimensionality configuration; bars grouped by contamination level.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams.update({
    'font.size': 11, 'axes.titlesize': 12, 'axes.labelsize': 11,
    'xtick.labelsize': 9, 'ytick.labelsize': 10, 'legend.fontsize': 9,
})

colors_6 = {
    'TB dense':    '#2166ac',
    'RTB dense':   '#d6604d',
    'CRTB dense':  '#1a9641',
    'TB sparse':   '#92c5de',
    'RTB sparse':  '#f4a582',
    'CRTB sparse': '#a6d96a',
}

methods_dense  = [('TB dense',   'tb_mse'),  ('RTB dense',  'rtb_mse'),  ('CRTB dense',  'crtb_mse')]
methods_sparse = [('TB sparse',  'tbs_mse'), ('RTB sparse', 'rtbs_mse'), ('CRTB sparse', 'crtbs_mse')]
all_methods = methods_dense + methods_sparse

cont_labels = [f"{int(c*100)}%" for c in cell_cont_pcts]

fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharey=False)

for ax_idx, (dim_label, (p_signal, p_noise)) in enumerate(dim_configs.items()):
    ax = axes[ax_idx // 3, ax_idx % 3]
    sub = df[df['dim_config'] == dim_label].copy()

    x = np.arange(len(cell_cont_pcts))
    n_methods = len(all_methods)
    width = 0.13

    for i, (label, col) in enumerate(all_methods):
        offset = (i - (n_methods - 1) / 2) * width
        vals = sub[f'{col}_mean'].values
        errs = sub[f'{col}_sd'].values
        ax.bar(x + offset, vals, width, yerr=errs, label=label,
               color=colors_6[label], edgecolor='white', linewidth=0.4, capsize=2)

    ax.set_xticks(x)
    ax.set_xticklabels(cont_labels)
    ax.set_xlabel('Cell contamination')
    ax.set_ylabel('MSE($\\mathbf{B}$)')
    ax.set_title(f'$p_{{\\rm signal}}$={p_signal}, $p_{{\\rm noise}}$={p_noise} (p={p_signal+p_noise})')
    if ax_idx == 0:
        ax.legend(loc='upper left', ncol=2)

fig.suptitle(f'MSE(B) under cellwise contamination — {n_repeats} repeats, n={n}', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('simulation_crtb_mse_bars.png', dpi=150, bbox_inches='tight')
plt.show()

## Heatmap: CRTB/RTB and CRTB/TB MSE ratios

Values < 1 (green) indicate CRTB outperforms the competitor.

In [ ]:
df['CRTB/TB_dense']    = (df['crtb_mse_mean']  / df['tb_mse_mean']).round(3)
df['CRTB/RTB_dense']   = (df['crtb_mse_mean']  / df['rtb_mse_mean']).round(3)
df['CRTBs/TBs_sparse'] = (df['crtbs_mse_mean'] / df['tbs_mse_mean']).round(3)
df['CRTBs/RTBs_sparse']= (df['crtbs_mse_mean'] / df['rtbs_mse_mean']).round(3)

df['scenario'] = df['cell_cont_pct'].apply(lambda c: f"{int(c*100)}% cells")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, col, title in [
    (axes[0], 'CRTB/RTB_dense',    'CRTB dense / RTB dense'),
    (axes[1], 'CRTBs/RTBs_sparse', 'CRTB sparse / RTB sparse'),
]:
    pivot = df.pivot_table(index='scenario', columns='dim_config', values=col)
    pivot = pivot[[c for c in dim_configs if c in pivot.columns]]

    col_labels = []
    for c in pivot.columns:
        ps_, pn = dim_configs[c]
        col_labels.append(f"p={ps_+pn}\n({ps_}+{pn})")

    im = ax.imshow(pivot.values, cmap='RdYlGn_r', aspect='auto', vmin=0.3, vmax=1.7)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(col_labels, fontsize=9)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=10)
    ax.set_title(title, fontsize=12)
    plt.colorbar(im, ax=ax, shrink=0.8)

    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f"{val:.2f}", ha='center', va='center', fontsize=9,
                        color='black' if 0.6 < val < 1.4 else 'white')

fig.suptitle('MSE ratio (< 1 = CRTB wins, green)', fontsize=13)
plt.tight_layout()
plt.savefig('simulation_crtb_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## MSE vs cell contamination percentage

Fixed configuration: p_signal=20, p_noise=80 (p=100).  
Cell contamination swept from 0 % to 35 % in 5 % steps.

In [ ]:
rng_sweep = np.random.default_rng(123)
cont_sweep = np.arange(0, 0.36, 0.05)
p_signal_sw, p_noise_sw = 20, 80
n_rep_sw = 200

sweep_methods = ['tb_mse', 'rtb_mse', 'crtb_mse', 'tbs_mse', 'rtbs_mse', 'crtbs_mse']
sweep_results = {m: {'mean': [], 'sd': []} for m in sweep_methods}

t0 = time.time()
for ccp in cont_sweep:
    collected = {m: [] for m in sweep_methods}
    for rep in range(n_rep_sw):
        res = run_single(rng_sweep, n, p_signal_sw, p_noise_sw, k, q,
                         sigma_e, sigma_f, ccp, outlier_shift, row_rate)
        for m in sweep_methods:
            collected[m].append(res.get(m, np.nan))
    for m in sweep_methods:
        sweep_results[m]['mean'].append(np.nanmean(collected[m]))
        sweep_results[m]['sd'].append(np.nanstd(collected[m]))
    print(f"  cont={ccp*100:.0f}%  "
          f"TB={sweep_results['tb_mse']['mean'][-1]:.4f}  "
          f"RTB={sweep_results['rtb_mse']['mean'][-1]:.4f}  "
          f"CRTB={sweep_results['crtb_mse']['mean'][-1]:.4f}  "
          f"({time.time()-t0:.0f}s)", flush=True)

print(f"Done in {time.time()-t0:.0f}s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sweep_labels = {
    'tb_mse':    ('TB dense',    colors_6['TB dense']),
    'rtb_mse':   ('RTB dense',   colors_6['RTB dense']),
    'crtb_mse':  ('CRTB dense',  colors_6['CRTB dense']),
    'tbs_mse':   ('TB sparse',   colors_6['TB sparse']),
    'rtbs_mse':  ('RTB sparse',  colors_6['RTB sparse']),
    'crtbs_mse': ('CRTB sparse', colors_6['CRTB sparse']),
}
baselines = {m: sweep_results[m]['mean'][0] for m in sweep_methods}

for ax, method_subset, title in [
    (axes[0], ['tb_mse', 'rtb_mse', 'crtb_mse'],        'Dense methods'),
    (axes[1], ['tbs_mse', 'rtbs_mse', 'crtbs_mse'],     'Sparse methods'),
]:
    for m in method_subset:
        label, color = sweep_labels[m]
        means = np.array(sweep_results[m]['mean'])
        sds   = np.array(sweep_results[m]['sd'])
        rel   = (means - baselines[m]) / max(baselines[m], 1e-10) * 100
        rel_sd = sds / max(baselines[m], 1e-10) * 100
        ax.plot(cont_sweep * 100, rel, 'o-', label=label, color=color, linewidth=2, markersize=5)
        ax.fill_between(cont_sweep * 100, rel - rel_sd, rel + rel_sd, alpha=0.15, color=color)
    ax.axhline(0, color='grey', linestyle='--', linewidth=0.8)
    ax.set_xlabel('Cell contamination (%)')
    ax.set_ylabel('Relative MSE increase (%)')
    ax.set_title(title)
    ax.legend()

fig.suptitle(f'Relative MSE increase vs cell contamination '
             f'($p_{{\\rm signal}}$={p_signal_sw}, $p_{{\\rm noise}}$={p_noise_sw})',
             fontsize=13)
plt.tight_layout()
plt.savefig('simulation_crtb_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

## Variable selection F1 score

Sparse methods only, on configurations that include uninformative variables.

In [ ]:
noise_configs = {k_: v for k_, v in dim_configs.items() if v[1] > 0}
n_panels = len(noise_configs)

fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 5), sharey=True)
if n_panels == 1:
    axes = [axes]

f1_methods = [
    ('TBs F1',   'tbs_f1',   colors_6['TB sparse']),
    ('RTBs F1',  'rtbs_f1',  colors_6['RTB sparse']),
    ('CRTBs F1', 'crtbs_f1', colors_6['CRTB sparse']),
]
x = np.arange(len(cell_cont_pcts))
width = 0.25

for ax_idx, (dim_label, (ps_, pn)) in enumerate(noise_configs.items()):
    ax = axes[ax_idx]
    sub = df[df['dim_config'] == dim_label]

    for i, (label, col, color) in enumerate(f1_methods):
        offset = (i - 1) * width
        ax.bar(x + offset, sub[f'{col}_mean'].values, width,
               yerr=sub[f'{col}_sd'].values, label=label,
               color=color, edgecolor='white', linewidth=0.5, capsize=3)

    ax.set_xticks(x)
    ax.set_xticklabels(cont_labels)
    ax.set_xlabel('Cell contamination')
    ax.set_ylim(0, 1.05)
    ax.set_title(f'$p_{{\\rm signal}}$={ps_}, $p_{{\\rm noise}}$={pn} (p={ps_+pn})')
    if ax_idx == 0:
        ax.set_ylabel('F1 score (X variable selection)')
    ax.legend(loc='lower left')

fig.suptitle(f'Variable selection F1 — {n_repeats} repeats, n={n}', fontsize=13)
plt.tight_layout()
plt.savefig('simulation_crtb_f1.png', dpi=150, bbox_inches='tight')
plt.show()

## Assertions

Sanity checks asserting that CRTB outperforms both TB and RTB under contamination,
and that all methods perform comparably at 0 % contamination.

In [ ]:
failures = []

for dim_label in dim_configs:
    sub = df[df['dim_config'] == dim_label]

    # At 0 % contamination CRTB should not be dramatically worse than TB
    row0 = sub[sub['cell_cont_pct'] == 0.0].iloc[0]
    ratio_clean = row0['crtb_mse_mean'] / max(row0['tb_mse_mean'], 1e-12)
    if ratio_clean > 3.0:
        failures.append(f"{dim_label} @ 0%: CRTB/TB ratio = {ratio_clean:.2f} (expected ≤ 3)")

    # At 20 % and 30 % contamination CRTB dense should beat RTB dense
    for ccp in [0.20, 0.30]:
        row = sub[sub['cell_cont_pct'] == ccp]
        if len(row) == 0:
            continue
        row = row.iloc[0]
        # CRTB should have lower or similar MSE than RTB
        if row['crtb_mse_mean'] > row['rtb_mse_mean'] * 1.5:
            failures.append(
                f"{dim_label} @ {int(ccp*100)}%: CRTB_mse={row['crtb_mse_mean']:.4f} "
                f"> 1.5 × RTB_mse={row['rtb_mse_mean']:.4f}"
            )

# F1: sparse CRTB should not have lower F1 than sparse TB at high contamination
for dim_label, (ps_, pn) in dim_configs.items():
    if pn == 0:
        continue
    sub = df[df['dim_config'] == dim_label]
    for ccp in [0.20, 0.30]:
        row = sub[sub['cell_cont_pct'] == ccp]
        if len(row) == 0:
            continue
        row = row.iloc[0]
        if row['crtbs_f1_mean'] < row['tbs_f1_mean'] - 0.20:
            failures.append(
                f"{dim_label} @ {int(ccp*100)}%: CRTBs F1={row['crtbs_f1_mean']:.3f} "
                f"< TBs F1={row['tbs_f1_mean']:.3f} - 0.20"
            )

if failures:
    print("ASSERTION FAILURES:")
    for f in failures:
        print(" ", f)
else:
    print("All assertions passed.")

## Conclusions

This simulation evaluates CRTB under the challenging scenario where **more than 50 %
of rows contain contaminated cells**, at three cell contamination levels (10 %, 20 %, 30 %).

Key questions addressed:
1. **Robustness** — does CRTB preserve coefficient accuracy better than TB and RTB
   as cell contamination increases?
2. **Variable selection** — do the sparse CRTB variants maintain good F1 scores
   under cellwise contamination that compromises TB-sparse and RTB-sparse?
3. **Clean-data efficiency** — is there a meaningful efficiency cost for CRTB at 0 %
   contamination?
4. **Dimensionality** — do the conclusions hold across both p < n and p > n regimes?